# RSNA Knee — Notebook 02: Conservative Fine-Tuning (FT-A)

This is the first actual continuation-training stage after Notebook 01.

Notebook 01 established:

- 4,407 training studies;
- only 58 studies have complete 12-target expert labels;
- the existing package contains 20 DINOv2 members (4 per original fold);
- the checkpoints contain model weights and epoch metadata but no optimizer/scheduler state;
- no original `StudyInstanceUID -> fold` artifact was verified.

Because the historical fold map is unavailable, this notebook **does not claim strict historical OOF**. Instead it creates a new deterministic 5-fold split only inside the 58 expert-labelled studies and uses it as a conservative promotion gate for fine-tuning.

### FT-A policy

- keep the original 0.891 package untouched;
- default: fine-tune only the best-holdout member from each original fold (5 new members);
- train the head + final 2 DINO transformer blocks;
- very small backbone learning rate;
- random overlapping 3-slice training windows;
- target-balanced masked BCE;
- early stopping / rollback: a fine-tuned state is promoted only if it improves validation BCE over its own base checkpoint;
- save new checkpoint fingerprints and an inference-compatible `manifest.json`.

### Required Kaggle inputs

1. RSNA Knee competition data.
2. The original `rsna-knee-weights` package used by the 0.89 notebook.
3. The same local DINOv2-small Hugging Face model input used by the old inference notebook.
4. Notebook 01 output (`rsna_stage1_bundle/` or `rsna_stage1_bundle.zip`).

The final output folder is `/kaggle/working/rsna_stage2_fta/`. Add this notebook's output as an input to Notebook 03.

In [ ]:
from __future__ import annotations

import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import hashlib
import json
import math
import re
import shutil
import time
import traceback
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import roc_auc_score

T0 = time.time()
SEED = int(os.environ.get("FT_SEED", "2026"))
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

# Conservative FT defaults.
FT_MEMBER_MODE = os.environ.get("FT_MEMBER_MODE", "champions").lower()
# champions = best checkpoint per original fold (recommended)
# top2      = best 2 per original fold
# all20     = every member
FT_UNFREEZE_LAST = int(os.environ.get("FT_UNFREEZE_LAST", "2"))
FT_MAX_EPOCHS = int(os.environ.get("FT_MAX_EPOCHS", "4"))
FT_PATIENCE = int(os.environ.get("FT_PATIENCE", "2"))
FT_HEAD_LR = float(os.environ.get("FT_HEAD_LR", "2e-4"))
FT_BACKBONE_LR = float(os.environ.get("FT_BACKBONE_LR", "1e-5"))
FT_WEIGHT_DECAY = float(os.environ.get("FT_WEIGHT_DECAY", "0.01"))
FT_BATCH = int(os.environ.get("FT_BATCH", "2"))
FT_GRAD_ACCUM = int(os.environ.get("FT_GRAD_ACCUM", "2"))
FT_MIN_DELTA = float(os.environ.get("FT_MIN_DELTA", "0.0005"))
FT_N_FOLDS = 5
EVAL_BATCH = int(os.environ.get("FT_EVAL_BATCH", "4"))
TIME_BUDGET = float(os.environ.get("FT_TIME_BUDGET_HOURS", "8")) * 3600

# Checkpoint-compatible pixel defaults. Manifest config overwrites these.
CROP_MM = 130.0
CACHE_IMG = 336
IMG = CACHE_IMG
GROUP = 3
CACHE_SLICES = 12
N_GROUP = max(CACHE_SLICES // GROUP, 1)
SLICE_BAND = (0.20, 0.80)

HDR_THREADS = int(os.environ.get("HDR_THREADS", "16"))
PIX_THREADS = int(os.environ.get("PIX_THREADS", "12"))
ORDER_THREADS = int(os.environ.get("ORDER_THREADS", "32"))
ORDER_BUDGET_S = 5400
LAT_MIN_OFFSET_MM = 20.0
LEGACY_LAT_OFFSET_MM = 5.0

RULES_NATIVE = {
    "order": "normal",
    "lat": "centre",
    "slot_fallback": False,
    "decode_fill": "nearest",
}
RULES_LEGACY = {
    "order": "dominant_axis",
    "lat": "corner_x",
    "slot_fallback": True,
    "decode_fill": "zero",
}
RULES = dict(RULES_NATIVE)

SLOTS_RECOVERED = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]
SLOTS_PUBLIC = [
    ("SAG_FLUID", "Sagittal", None, True),
    ("COR_FLUID", "Coronal", None, True),
    ("AX_FLUID", "Axial", None, True),
    ("SAG_STRUCT", "Sagittal", None, False),
    ("COR_STRUCT", "Coronal", None, False),
    ("AX_STRUCT", "Axial", None, False),
]
SLOTS = SLOTS_RECOVERED
N_SLOT = len(SLOTS)

POOL_PARTS = {"cls_mean": 2, "cls_mean_focal": 3}
SLOT_PRIOR_TABLE = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}
SLOT_PRIOR_STRENGTH = 0.55

FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(
    r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
    r"water excit|\btirm\b|\bsting\b|\bfatsup\b"
)
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")

FRONTIER_TARGET_POOL = {
    "Fracture": "max",
    "Contusion": "max",
    "Medial Meniscus": "max",
    "Lateral Meniscus": "max",
    "ACL": "top2",
    "MCL": "top2",
    "Baker's": "max",
}

OUT = Path("/kaggle/working/rsna_stage2_fta")
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


class WeightsError(RuntimeError):
    pass


def find_train_root():
    explicit = os.environ.get("KNEE_INPUT_DIR", "").strip()
    candidates = [Path(explicit)] if explicit else []
    candidates += [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
    ]
    for c in candidates:
        if (c / "train.csv").is_file() and (c / "train_series").is_dir():
            return c

    base = Path("/kaggle/input")
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            root = Path(root)
            if (root / "train.csv").is_file() and (root / "train_series").is_dir():
                return root
    raise FileNotFoundError("RSNA Knee competition training mount not found")


def _valid_weight_package(root: Path):
    root = Path(root)
    p = root / "manifest.json"
    if not p.is_file():
        return False
    try:
        man = json.loads(p.read_text())
    except Exception:
        return False
    members = man.get("members")
    if not isinstance(members, list) or not members:
        return False
    return all((root / str(m.get("file"))).is_file() for m in members)


def find_weights():
    explicit = os.environ.get("KNEE_WEIGHTS_DIR", "").strip()
    candidates = [Path(explicit)] if explicit else []
    candidates += [
        Path("/kaggle/input/rsna-knee-weights"),
        Path("/kaggle/input/datasets/pilkwang/rsna-knee-weights"),
    ]
    for p in candidates:
        if p and _valid_weight_package(p):
            return p

    base = Path("/kaggle/input")
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            if "manifest.json" in files and _valid_weight_package(Path(root)):
                # Avoid accidentally picking Stage 2 output from a prior run.
                man = json.loads((Path(root) / "manifest.json").read_text())
                if not any("ft_stage" in m for m in man.get("members", [])):
                    return Path(root)
    raise FileNotFoundError("Original rsna-knee-weights package not found")


def _looks_like_dinov2_config(config_path: Path):
    try:
        cfg = json.loads(config_path.read_text())
    except Exception:
        return False
    text = json.dumps(cfg).lower()
    return (
        cfg.get("model_type", "").lower() == "dinov2"
        or "dinov2" in text
        or any("dinov2" in str(x).lower() for x in cfg.get("architectures", []) or [])
    )


def find_dinov2(variant="small"):
    explicit = os.environ.get("KNEE_DINOV2_DIR", "").strip()
    if explicit:
        p = Path(explicit)
        if (p / "config.json").is_file():
            return p
        raise FileNotFoundError(f"KNEE_DINOV2_DIR={explicit!r} has no config.json")

    base = Path("/kaggle/input")
    strong, weak = [], []
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            if "config.json" not in files:
                continue
            p = Path(root)
            if _looks_like_dinov2_config(p / "config.json"):
                strong.append(p)
            elif "dinov2" in str(p).lower():
                weak.append(p)

    hits = strong or weak
    if not hits:
        return None

    target_hidden = {"small": 384, "base": 768, "large": 1024, "giant": 1536}.get(str(variant).lower())
    if target_hidden is not None:
        for p in hits:
            try:
                cfg = json.loads((p / "config.json").read_text())
                if int(cfg.get("hidden_size", -1)) == target_hidden:
                    return p
            except Exception:
                pass
    return hits[0]


def find_stage1():
    explicit = os.environ.get("STAGE1_DIR", "").strip()
    if explicit:
        p = Path(explicit)
        if (p / "stage1_status.json").is_file():
            return p

    # Prefer already-unpacked Notebook 01 output.
    base = Path("/kaggle/input")
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            if "stage1_status.json" in files and "expert_complete_12.csv" in files:
                return Path(root)

    # Or extract the compact ZIP produced by Notebook 01.
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            if "rsna_stage1_bundle.zip" in files:
                z = Path(root) / "rsna_stage1_bundle.zip"
                dest = Path("/kaggle/working/stage1_input")
                dest.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(z) as fh:
                    fh.extractall(dest)
                if (dest / "stage1_status.json").is_file():
                    return dest

    raise FileNotFoundError(
        "Notebook 01 output not found. Add Notebook 01 output as a Kaggle input "
        "or set STAGE1_DIR."
    )


ROOT = find_train_root()
WEIGHTS = find_weights()
STAGE1 = find_stage1()
DINO = find_dinov2("small")

log(f"competition root: {ROOT}")
log(f"base weights: {WEIGHTS}")
log(f"stage1 input: {STAGE1}")
log(f"DINOv2: {DINO}")

if DINO is None:
    raise FileNotFoundError(
        "DINOv2-small Hugging Face model directory not found. "
        "Attach the same DINO input used by the old notebook."
    )

## 1. Validate Notebook 01 handoff

No historical fold map was found in your actual Notebook 01 run. Therefore this stage creates a **new expert-CV split** and records that fact in every checkpoint. It does not relabel this split as the original folds.

In [ ]:
stage1_status = json.loads((STAGE1 / "stage1_status.json").read_text())
resume_manifest = json.loads((STAGE1 / "resume_manifest.json").read_text())
expert = pd.read_csv(STAGE1 / "expert_complete_12.csv", dtype={"StudyInstanceUID": str})

if not stage1_status.get("stage1_ok", False):
    raise RuntimeError("Stage 1 did not report stage1_ok=True")

if len(expert) < 20:
    raise RuntimeError(f"Expected the expert-labelled subset; found only {len(expert)} rows")

missing = [t for t in TARGETS if t not in expert.columns]
if missing:
    raise KeyError(f"expert_complete_12.csv missing targets: {missing}")

if expert[TARGETS].isna().any().any():
    raise RuntimeError("Notebook 02 expects complete expert labels in expert_complete_12.csv")

print("Stage 1 historical fold map verified:", stage1_status.get("has_verified_original_fold_map"))
print("Expert studies:", len(expert))
print("Fine-tuning validation mode: NEW_EXPERT_CV (not historical OOF)")

## 2. Load manifest, select FT-A starting members, and auto-match the slot scheme

In [ ]:
base_manifest = json.loads((WEIGHTS / "manifest.json").read_text())
members = base_manifest["members"]

def choose_members(members, mode):
    frame = pd.DataFrame([
        {
            "idx": i,
            "id": m.get("id"),
            "fold": int(m.get("fold")),
            "holdout": float(m.get("holdout") or -1),
        }
        for i, m in enumerate(members)
    ])
    frame = frame.sort_values(["fold", "holdout"], ascending=[True, False])

    if mode == "champions":
        keep = frame.groupby("fold", as_index=False).head(1)
    elif mode == "top2":
        keep = frame.groupby("fold", as_index=False).head(2)
    elif mode == "all20":
        keep = frame
    else:
        raise ValueError("FT_MEMBER_MODE must be champions, top2, or all20")

    return [members[int(i)] for i in keep["idx"].tolist()]


selected_members = choose_members(members, FT_MEMBER_MODE)
print("FT_MEMBER_MODE:", FT_MEMBER_MODE)
print("selected members:", len(selected_members))

display(pd.DataFrame([
    {"id": m["id"], "base_fold": m.get("fold"), "holdout": m.get("holdout"), "file": m["file"]}
    for m in selected_members
]).sort_values(["base_fold", "holdout"], ascending=[True, False]))


def slot_scheme_for_names(names):
    names = list(names)
    rec = [x[0] for x in SLOTS_RECOVERED]
    pub = [x[0] for x in SLOTS_PUBLIC]
    if names == rec:
        return SLOTS_RECOVERED
    if names == pub:
        return SLOTS_PUBLIC
    raise WeightsError(f"Unknown checkpoint slot scheme: {names}")


def adopt_config_globals(cfg):
    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM
    global SLICE_BAND, RULES, SLOTS, N_SLOT

    CACHE_IMG = IMG = int(cfg["img"])
    GROUP = int(cfg["group"])
    CACHE_SLICES = int(cfg["slices"])
    N_GROUP = max(CACHE_SLICES // GROUP, 1)
    CROP_MM = float(cfg["crop_mm"])
    SLICE_BAND = tuple(float(x) for x in cfg["band"])

    rules = cfg.get("rules") or RULES_NATIVE
    unknown = {
        k: v for k, v in rules.items()
        if k not in RULES_NATIVE or v not in (RULES_NATIVE[k], RULES_LEGACY[k])
    }
    if unknown:
        raise WeightsError(f"unsupported pixel rules in manifest: {unknown}")
    RULES = {**RULES_NATIVE, **rules}

    SLOTS = slot_scheme_for_names(cfg["slots"])
    N_SLOT = len(SLOTS)


pixel_groups = {}
for m in selected_members:
    pixel_groups.setdefault(m["pixel_group"], []).append(m)

print("selected pixel groups:", len(pixel_groups))
for k, gm in pixel_groups.items():
    cfg = json.loads(k)
    print(
        f"- {cfg['img']}px, slices={cfg['slices']}, group={cfg['group']}, "
        f"crop={cfg['crop_mm']}mm -> {len(gm)} selected member(s)"
    )

## 3. Reuse the checkpoint-compatible MRI preprocessing

The following functions come from the old inference pipeline so FT-A sees the same sequence selection, slice ordering, crop, intensity normalization, and laterality convention as the checkpoint it starts from.

In [ ]:
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing", "Rows",
            "Columns", "RescaleSlope", "RescaleIntercept",
            # Position and orientation are read from the same header probe() already
            # opens, so they cost nothing, and they are what recovers the side when the
            # Laterality tag is absent - which it is for half the studies here.
            "ImagePositionPatient", "ImageOrientationPatient"]


def _hdr_vec(s, n):
    """Parse a DICOM multi-value string as stored by probe(): floats joined by `|`."""
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split("|")]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None


def side_from_geometry(h):
    """Study -> 'L' / 'R' / None, from where the image sits in the patient.

    `Laterality` (0020,0060) is Type 2C and may legitimately be absent; in this corpus it
    is missing on exactly half the studies, and the vendors it is missing from are whole
    vendors rather than scattered series. A study with no tag is not a left knee, but the
    normalisation upstream treats it as one, so half the corpus was never normalised and
    the five side-defined targets - the two menisci, the two tibiofemoral compartments
    and the medial collateral ligament - saw that axis reversed on a large minority of it.

    The patient coordinate system fixes this without the tag: +x is the patient's left, so
    the centre of a right knee sits at negative x. The centre is used rather than
    `ImagePositionPatient` itself because that is the corner of the image, which is offset
    by half a field of view - enough to change the sign on a knee near the midline.

    The median over a study's series is what is thresholded, not a single series: probe()
    reads one arbitrary slice per series, which on a sagittal stack can sit anywhere
    across the joint. Studies whose centre falls near the midline are left unresolved
    rather than guessed - measured against the tagged half, the rule is right 97% of the
    time overall and no better than chance inside 20 mm.
    """
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
        iop = _hdr_vec(getattr(r, "ImageOrientationPatient", None), 6)
        ps = _hdr_vec(getattr(r, "PixelSpacing", None), 2)
        rows, cols = getattr(r, "Rows", None), getattr(r, "Columns", None)
        if ipp is None or iop is None or ps is None or not rows or not cols:
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else ("R" if m < 0 else "L")
    return out


def side_from_corner_x(h):
    """The laterality an imported member was fitted under.

    It thresholds the median raw `ImagePositionPatient` x over a study's series. That is
    the x of the image *corner*, not of its centre, so it differs from the rule above by
    up to half a field of view - which is enough to reverse the sign on a knee scanned
    near the midline. The dead zone is 5 mm rather than 20 mm, so it also commits on
    studies the rule above leaves unresolved.

    Neither difference changes a shape. Each one decides whether a study is mirrored, and
    a study mirrored one way at training and the other at inference presents the five
    side-defined targets with their axis reversed.
    """
    out = {}
    for st, g in h.groupby("StudyInstanceUID"):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        # DICOM patient coordinates are LPS: +x is the patient's left.
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else ("R" if x < 0 else "L")
    return out


def lat_of(h, tag=""):
    """Study -> 'L' / 'R' / None: the tag where it exists, geometry where it does not.

    The tag is present on exactly half the studies here and is sometimes an empty
    string rather than absent, which is not the same as NaN. Treating the other half
    as left-sided is what `normalise_laterality` did by omission, so the geometry
    fallback is not a refinement - it is the difference between normalising half the
    corpus and normalising all of it.
    """
    geo = side_from_corner_x(h) if RULES["lat"] == "corner_x" else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = {}, 0, 0, 0, 0
    for st, g in h.groupby("StudyInstanceUID"):
        v = [str(x).strip().upper() for x in g["Laterality"].dropna()]
        if RULES["lat"] == "corner_x" and "ImageLaterality" in g.columns:
            # The legacy rule reads the second tag too, so a study tagged only there is
            # resolved from the tag rather than from geometry.
            v += [str(x).strip().upper() for x in g["ImageLaterality"].dropna()]
        v = [x[0] for x in v if x and x[0] in ("L", "R")]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f"{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, "
        f"{n_none} unresolved; tag and geometry disagree on {n_disagree} "
        f"({n_disagree / max(n_tag, 1):.1%} of the tagged)")
    return d



def probe(item):
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
           "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def walk(split):
    """Every series directory of a split, with one header read per series.

    An absent split returns an empty frame *with the columns annotate expects*. Returning
    a bare DataFrame looks like the same thing and is not: the next call indexes
    `SeriesDescription` and raises KeyError, so the branch that exists to survive a
    missing split is what turns it into a crash.
    """
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=["split", "StudyInstanceUID", "SeriesInstanceUID",
                                     "dir", "files", "n_slices"] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)


def annotate(df):
    """Recover fat suppression and pulse-sequence weighting from the header."""
    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    # GE writes SAT_GEMS for spatial saturation, so ScanOptions must be matched as
    # exact tokens; a substring test on "SAT" fires on non-fat-sat series.
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)

    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df

In [ ]:
def walk_selected(split, study_ids):
    study_ids = {str(x) for x in study_ids}
    base = ROOT / split
    items = []

    if not base.is_dir():
        raise FileNotFoundError(base)

    for study_id in sorted(study_ids):
        study_dir = base / study_id
        if not study_dir.is_dir():
            continue
        for series in os.scandir(study_dir):
            if series.is_dir():
                items.append((split, study_id, series.name, series.path))

    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))

    out = pd.DataFrame(rows)
    if out.empty:
        raise RuntimeError(f"No DICOM series found for {len(study_ids)} selected studies")
    return out

In [ ]:
def _as_bool(v):
    if pd.isna(v):
        return None
    s = str(v).strip().upper()
    if s in {"1", "TRUE", "T", "YES", "Y"}:
        return True
    if s in {"0", "FALSE", "F", "NO", "N"}:
        return False
    try:
        f = float(s)
        return True if f == 1 else False if f == 0 else None
    except Exception:
        return None


def audit_official_sequence_metadata(inferred, official):
    """Audit only. Do not change imported checkpoint pixels from this table."""
    need = {"SeriesInstanceUID", "Fluid_Sensitive", "Fat_Suppression"}
    if inferred.empty or official.empty or not need.issubset(official.columns):
        return
    a = inferred[["SeriesInstanceUID", "fluid", "fatsat"]].copy()
    b = official[["SeriesInstanceUID", "Fluid_Sensitive", "Fat_Suppression"]].copy()
    b["official_fluid"] = b["Fluid_Sensitive"].map(_as_bool)
    b["official_fatsat"] = b["Fat_Suppression"].map(_as_bool)
    m = a.merge(b[["SeriesInstanceUID", "official_fluid", "official_fatsat"]],
                on="SeriesInstanceUID", how="inner")
    for inferred_col, official_col, name in [
        ("fluid", "official_fluid", "Fluid_Sensitive"),
        ("fatsat", "official_fatsat", "Fat_Suppression"),
    ]:
        valid = m[official_col].notna() & m[inferred_col].notna()
        if valid.any():
            agree = (m.loc[valid, inferred_col].astype(bool).values ==
                     m.loc[valid, official_col].astype(bool).values).mean()
            log(f"metadata audit {name}: {agree:.1%} agreement on {int(valid.sum())} series")

In [ ]:
def pick_slots(series_df, plane_map):
    """One series per slot per study.

    Ties are broken toward the stack with the most slices: a thicker stack samples the
    joint more densely, and the three-slice sampler below benefits from the margin.
    """
    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs)
            # fluid=None means "do not condition on weighting" - the public scheme,
            # where the single provided flag stands in for both axes at once.
            if fluid is not None:
                sel &= (g["fluid"] == fluid)
            cand = g[sel]
            # A slot with no series matching its predicate stays empty, and no substitute
            # is admitted from a neighbouring predicate. Relaxing the weighting to fill a
            # T1 slot would draw from the pool `SAG_FLUID_NOFS` selects from, since that
            # pool is what remains once the weighting is dropped: over the training corpus
            # it would put one series in two slots for 2383 of 4407 studies and leave 56%
            # of the T1 slot holding PD or T2. The presence mask would then assert a
            # sequence that was never acquired, and the per-diagnosis softmax of §6 would
            # divide its attention across two identical slots, giving one acquisition
            # about twice the weight it carries in a study that holds both. The mask is
            # there to say a slot is absent, which is what an absent slot is.
            if len(cand) == 0 and RULES["slot_fallback"] and fluid is False:
                # The relaxation the paragraph above rejects, reproduced because an
                # imported member was fitted with its T1 slots filled this way: over half
                # of that member's training studies had a T1 slot holding a series that
                # is not T1. Leaving those slots empty would present it with a presence
                # mask it never saw.
                cand = g[(g["plane"] == plane) & (~g["fatsat"])]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out

In [ ]:
ORDER_TAGS = [(0x0020, 0x0032), (0x0020, 0x0037), (0x0020, 0x0013)]

# Series in which at least one sampled slice would not decode. A list rather than a
# counter because appending is atomic under the reader threads, and reported rather than
# swallowed: unreported, a decode failure is indistinguishable from a black knee.
DECODE_FAILED = []


def cache_tag(rules=None):
    """The name a decoded cache is stored under.

    It has to name everything that decides the pixels, not only their dimensions. Two
    configurations that agree on resolution, slice count, crop and band but disagree on
    how a slice is chosen produce different arrays of identical shape - so a tag built
    from the dimensions alone lets the second attach to the first one's file and train
    against pixels it never asked for, with nothing anywhere reporting a mismatch.

    A native reading keeps the plain name, so caches decoded before the rules existed
    stay valid; anything else earns a suffix.
    """
    r = dict(RULES if rules is None else rules)
    t = (f"{CACHE_IMG}px_{CACHE_SLICES}sl_{int(CROP_MM)}mm_"
         f"{SLICE_BAND[0]:.2f}-{SLICE_BAND[1]:.2f}")
    if {k: r.get(k, v) for k, v in RULES_NATIVE.items()} != RULES_NATIVE:
        t += "_" + hashlib.md5(json.dumps(r, sort_keys=True).encode()).hexdigest()[:6]
    return t


def _natural_key(name):
    return tuple(int(x) if x.isdigit() else x.lower()
                 for x in re.split(r"(\d+)", str(name)))


def _order_dominant_axis(rec):
    """The slice order an imported member was fitted under.

    It sorts on the raw patient coordinate along whichever axis varies most across the
    stack, rather than on the projection onto the slice normal. The two differ by a sign,
    not by a formula: measured over this corpus every sagittal series has a slice normal
    with n_x in [-1.00, -0.98], so p.n is the negative of the raw x this sorts on and the
    two stacks come out exactly reversed. Because the band sampler truncates rather than
    rounds, its nine indices are not symmetric about the middle, so nine slices drawn from
    a twenty-six slice stack under one order share two with the other.

    Missing geometry falls back to `InstanceNumber` and then to a natural sort of the file
    name, both at the same 80% threshold the imported pipeline used.
    """
    files, d = rec["files"], rec["dir"]
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=["ImagePositionPatient", "InstanceNumber"])
            raw = getattr(ds, "ImagePositionPatient", None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, "InstanceNumber", None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))

    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare,
                                 r[2] if r[2] is not None else float("inf"), r[3]))
    elif sum(r[2] is not None for r in rows) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float("inf"), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return [r[0] for r in rows], True


def order_slices(rec):
    """Return the series' files sorted along the through-plane axis.

    A DICOM file name here is a SOP Instance UID, which is assigned arbitrarily. Sorting
    by it therefore produces an order uncorrelated with anatomy - measured over one
    series, Spearman between file-name rank and physical position is 0.009, i.e. none.
    Anything that assumes the file order means something is then operating on noise: the
    three channels of a "2.5D" input are three unrelated views rather than neighbouring
    slices, "the middle of the stack" is a random subset, and reversing slice order to
    normalise laterality reverses nothing meaningful.

    The physical order is recoverable exactly. Each slice carries its position in patient
    coordinates and the in-plane axes; projecting the position onto the slice normal
    gives a signed through-plane coordinate, monotonic along the stack:

        n = r_x  x  r_y ,      k = p . n

    `InstanceNumber` is the fallback. It usually tracks the projection up to sign, but
    interleaved and multi-echo acquisitions need not number slices in the order they
    occupy in space - but the projection is signed in patient
    coordinates, which is what laterality normalisation needs.
    """
    if RULES["order"] == "dominant_axis":
        return _order_dominant_axis(rec)
    files, d = rec["files"], rec["dir"]
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any(k is None for k, _ in keyed):
        # A series with no usable geometry keeps its arbitrary order; that is worse than
        # sorting but better than dropping the series, and it is logged as a count.
        return files, False
    return [f for _, f in sorted(keyed, key=lambda t: t[0])], True


def read_slot(rec, n_slice=None, out_size=None):
    """`n_slice` physically spread slices from one series, at `out_size` pixels.

    Returns uint8 [n_slice, out, out] normalised per-series to its 1st-99th
    percentile. Percentiles rather than min/max because MR intensity has no absolute
    scale and a single bright vessel would otherwise compress the whole dynamic range.

    Reading is the expensive half of this pipeline, so the caller reads once at the
    largest configuration it needs and derives the smaller ones from the returned buffer
    rather than re-reading.
    """
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = rec.get("ordered") or rec["files"], rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None
    # Spread the samples over a central band of the stack: the outermost slices of a knee
    # series are mostly soft tissue outside the joint. The band is a constant rather than
    # a literal because how much of the stack is worth reading depends on how many slices
    # are being taken - at three the middle is all that fits, while at sixteen the ends
    # are worth having, and a Baker cyst sits at the posteromedial end of a sagittal one.
    lo, hi = int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])

    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None                      # no shape is known here; see below
        planes.append(a)

    # A slice that would not decode has no shape of its own, and inventing one is how a
    # single unreadable file erases a whole series: a substitute allocated at the resize
    # target while the decoded slices are still native makes the shape check below take
    # the substitute as the authority and zero the good slices with it, leaving a black
    # slot that the presence mask still reports as acquired.
    #
    # A failure is instead filled from the nearest slice that did decode - the same
    # convention the sampler already uses when the band holds fewer distinct slices than
    # were asked for - and a series where nothing decodes is reported absent, which the
    # mask can express, rather than black, which it cannot.
    got = [k for k, p in enumerate(planes) if p is not None]
    if RULES["decode_fill"] == "zero":
        # What an imported member was fitted with: a failure becomes a zero plane at the
        # resize target, which the shape check below then propagates to the whole slot.
        # It is the behaviour the paragraph above describes and rejects, kept here only
        # because that member's weights were learned against slots blacked out this way.
        if not got:
            DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p
                  for p in planes]
        got = list(range(len(planes)))
    if not got:
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]

    # Slices of one series can still differ in matrix size - multi-echo and some
    # reformats do - and those are genuinely not stackable.
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)

    # constant physical extent, then resize: PixelSpacing varies 3.4x across the corpus
    if px and np.isfinite(px) and px > 0:
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    # uint8, not float32. These buffers queue up between the reader threads and the
    # encoder, and at this size a float32 slot-series is several megabytes. Intensity is
    # already normalised into [0, 1] here, so eight bits cost nothing that a bilinear
    # resize has not already cost, and the queue is a quarter the size.
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)

In [ ]:
def normalise_laterality(img, plane, lat):
    """Map every knee onto a left-knee convention.

    Coronal and axial views mirror under a horizontal flip. Sagittal stacks are not
    mirror images of each other - the slice order runs medial-to-lateral in opposite
    directions - so the channel order is reversed instead.
    """
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])

In [ ]:
# Where the geometric slice order may be remembered between runs. Unset on the platform,
# because each run gets a fresh machine and there is nothing to remember; set off it,
# where the same corpus is cached again at every resolution and slice count and the order
# is a function of neither. It is opt-in so that the scored run's behaviour is decided by
# the code rather than by whether a file happens to be lying about.
ORDER_CACHE = os.environ.get("RSNA_ORDER_CACHE") or None


def build_cache(slot_map, plane_map, lat_map, tag):
    """Decode every (study, slot) once into an in-memory uint8 array.

    Fine-tuning revisits the same pixels every epoch. Reading them from the mount each
    time would make the epoch count a function of I/O rather than of learning, so they
    are decoded once and held as bytes: intensity has already been normalised into
    [0, 1], and eight bits cost nothing a bilinear resize has not already cost.

    CACHE_SLICES positions are kept per slot, which the training loop reads as N_GROUP
    groups of GROUP consecutive channels.
    """
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    n_job = len(jobs)

    # Ordering first, and as its own pass. It reads one header per slice of every chosen
    # series - far more file opens than the decode that follows - and on a network mount
    # that is latency, not work, so it gets its own wider pool.
    t_ord = time.time()
    n_slice_total = sum(len(j[3]["files"]) for j in jobs)
    log(f"{tag}: ordering {len(jobs)} slot-series ({n_slice_total} slice headers)")
    ok = done = 0
    CHUNK_O = 1024

    # A remembered order, when one is offered. The projection depends on the DICOM
    # geometry alone, so it is the same at every resolution and every slice count, and
    # it costs one header read per slice - the largest single cost in this pass. An entry
    # is validated by the number of files present, so a tree that has changed under it is
    # recomputed rather than trusted: order is derived data, and a stale entry would be
    # invisible in the way that matters most.
    seen = {}
    if ORDER_CACHE and Path(ORDER_CACHE).is_file():
        try:
            import json as _json
            seen = _json.loads(Path(ORDER_CACHE).read_text())
        except (OSError, ValueError):
            seen = {}
        hit = 0
        for _, _, _, rec in jobs:
            e = seen.get(rec["SeriesInstanceUID"])
            if e and len(e["files"]) == len(rec["files"]):
                rec["ordered"] = e["files"]
                ok += int(e["good"])
                hit += 1
        jobs = [j for j in jobs if "ordered" not in j[3]]
        log(f"{tag}: {hit} slot-series ordered from {ORDER_CACHE}, {len(jobs)} to read")

    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK_O):
            block = jobs[c0:c0 + CHUNK_O]
            for (_, _, _, rec), (files, good) in zip(
                    block, pool.map(lambda j: order_slices(j[3]), block)):
                rec["ordered"] = files
                ok += int(good)
                done += 1
                if ORDER_CACHE:
                    seen[rec["SeriesInstanceUID"]] = {"files": files, "good": bool(good)}
            # The ceiling is whichever comes first: the pass's own budget, or the share
            # of what is left of the run that it may take. The second is what makes the
            # first safe to set generously - a mount slow enough to matter cannot spend
            # the training time, because the budget shrinks as the run does.
            budget = min(ORDER_BUDGET_S, max(60.0, (TIME_BUDGET - (time.time() - T0)) * 0.35))
            if time.time() - t_ord > budget:
                log(f"{tag}: ordering budget spent at {done}/{len(jobs)}; "
                    f"the rest keep file order")
                break
    if ORDER_CACHE and done:
        import json as _json
        _t = Path(ORDER_CACHE).with_suffix(".tmp")
        _t.write_text(_json.dumps(seen))
        _t.replace(Path(ORDER_CACHE))
    log(f"{tag}: ordered {ok}/{n_job} by geometry "
        f"({n_job - ok} kept arbitrary) in {time.time() - t_ord:.0f}s")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    log(f"{tag}: decoding {len(jobs)} slot-series")
    n_failed_before = len(DECODE_FAILED)

    CHUNK = 512
    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            block = jobs[c0:c0 + CHUNK]
            for (st, k, plane, _), img in zip(
                    block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane,
                                                          lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < CHUNK:
                log(f"  {tag} {done}/{len(jobs)}")
            if time.time() - T0 > TIME_BUDGET:
                log(f"  {tag}: time budget reached during decode")
                break
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f"{tag}: {int(mask.sum())}/{len(jobs)} slots filled"
        + (f"; {n_failed} series had a slice that would not decode" if n_failed else ""))
    gc.collect()
    return studies, cache, mask

## 4. Checkpoint-compatible DINOv2 model

In [ ]:
class SlotHead(nn.Module):
    """Per-diagnosis attention over the slot embeddings of one study.

    Each finding is read on particular sequences - cruciates sagittally, collateral
    ligaments and the meniscal body coronally, patellar cartilage axially - so pooling
    the slots identically would dilute the one that carries the evidence with the rest.

    The aggregation is deliberately this simple. With a study-level label there is no
    signal telling the model which part of a study matters, so extra attention
    parameters below the slot level would have nothing to learn from and would spend
    their capacity fitting noise.
    """

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        # An imported member carries a fixed per-(diagnosis, slot) tilt on the attention
        # logits, set from the anatomy table below rather than learned. It is a buffer, so
        # it travels in the state dict and must exist for that member to load; exp(0.55)
        # gives a preferred slot about 1.73x the weight of an unpreferred one, which
        # biases the softmax without ever excluding a slot.
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and n_out == len(TARGETS):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer("slot_prior", p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias

In [ ]:
class Model(nn.Module):
    """Encoder plus head, trained end to end.

    A study arrives as a bag of slot images. The bag is flattened for the encoder and
    folded back before the head, so the encoder never sees the study structure and the
    head never sees pixels.
    """

    def __init__(self, backbone, dim, pool="cls_mean", prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            # The cache is held at the highest resolution any configuration needs; the
            # rest downsample from it, so every configuration sees the same pixels
            # through a different sampling grid rather than a different crop.
            x = F.interpolate(x, size=(img_size, img_size), mode="bilinear",
                              align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == "cls_mean_focal":
            # The upper tail of each channel over the patch grid, taken per channel
            # rather than by selecting whole patches: a finding occupies a small part of
            # the field, so a plain mean over 256 patches dilutes it by two orders of
            # magnitude, and this keeps the top eighth of each channel's responses.
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)

In [ ]:
def build_model(unfreeze_last, source=None, variant="small", pool="cls_mean", prior=False):
    from transformers import AutoModel

    p = Path(source) if source is not None else DINO
    if p is None:
        raise FileNotFoundError("DINOv2 weights not attached")

    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)

    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - int(unfreeze_last)):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True

    dim = bb.config.hidden_size
    trainable = sum(p.numel() for p in bb.parameters() if p.requires_grad)
    log(
        f"backbone: {n_layer} blocks, last {unfreeze_last} trainable "
        f"({trainable / 1e6:.1f}M backbone params)"
    )
    return Model(bb, dim, pool=pool, prior=prior)

## 5. Fingerprint contract + frontier validation prediction

In [ ]:
FINGERPRINT_TOL = 2e-3

def fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    n_slot = N_SLOT if n_slot is None else n_slot
    group = GROUP if group is None else group
    seed = SEED if seed is None else seed

    g = torch.Generator().manual_seed(seed)
    imgs = torch.randint(
        0, 256, (2, n_slot, group, img_size, img_size),
        generator=g, dtype=torch.uint8
    ).to(dev)
    mask = torch.ones(2, n_slot, device=dev)
    mask[1, -1] = 0.0

    was_training = model.training
    model.eval()
    with torch.no_grad():
        out = model(imgs, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out


def check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=""):
    got = fingerprint(model, dev, img_size)
    exp = np.asarray(expected, np.float32)
    if got.shape != exp.shape:
        raise WeightsError(f"{tag}fingerprint shape {got.shape} != stored {exp.shape}")
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(
            f"{tag}fingerprint differs by {d:.4g} > {tol:g}; "
            "architecture/preprocessing contract moved"
        )
    log(f"{tag}base fingerprint matches within {d:.2g}")
    return d


def window_starts(n_slice, group):
    if n_slice >= group:
        return list(range(n_slice - group + 1))
    return [0]


def _apply_frontier_pool(probs):
    v = probs.mean(dim=0)
    target_idx = {t: j for j, t in enumerate(TARGETS)}
    for target, mode in FRONTIER_TARGET_POOL.items():
        j = target_idx[target]
        x = probs[:, :, j]
        if mode == "max":
            v[:, j] = x.max(dim=0).values
        elif mode.startswith("top"):
            k = min(int(mode[3:]), x.shape[0])
            v[:, j] = x.topk(k, dim=0).values.mean(dim=0)
        elif mode == "mean":
            v[:, j] = x.mean(dim=0)
        else:
            raise ValueError(mode)
    return v


@torch.no_grad()
def predict_member_frontier(model, cache, mask, idx, dev, img_size):
    starts = window_starts(cache.shape[2], GROUP)
    model.eval()
    pred = np.empty((len(idx), len(TARGETS)), np.float32)

    for b0 in range(0, len(idx), EVAL_BATCH):
        sel = np.asarray(idx[b0:b0 + EVAL_BATCH], dtype=int)
        b1 = b0 + len(sel)
        m = torch.from_numpy(mask[sel]).to(dev)
        win = []

        for st in starts:
            rows = torch.from_numpy(
                np.ascontiguousarray(cache[sel, :, st:st + GROUP])
            ).to(dev)
            with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
                logits = model(rows, m, img_size).float()
            win.append(torch.sigmoid(logits))

        probs = torch.stack(win, dim=0)
        pred[b0:b1] = _apply_frontier_pool(probs).cpu().numpy()

    return pred

## 6. Build a new deterministic 5-fold split inside the 58 expert labels

This is **FT-stage CV**, not the original model's historical CV. The split is greedily balanced across the 12 positive labels so rare findings are spread as evenly as possible.

In [ ]:
Y_expert = expert[TARGETS].to_numpy(np.float32)

def make_balanced_multilabel_folds(y, n_folds=5, seed=2026):
    y = np.asarray(y, np.float32)
    n, t = y.shape
    rng = np.random.default_rng(seed)

    total_pos = y.sum(axis=0)
    rarity = 1.0 / np.maximum(total_pos, 1.0)
    priority = (y * rarity).sum(axis=1) + 0.01 * y.sum(axis=1)
    priority += rng.uniform(0, 1e-6, size=n)
    order = np.argsort(-priority)

    target_pos = total_pos / n_folds
    target_size = n / n_folds
    fold_pos = np.zeros((n_folds, t), np.float64)
    fold_size = np.zeros(n_folds, np.int64)
    assignment = np.full(n, -1, np.int64)

    max_size = int(math.ceil(n / n_folds))

    for i in order:
        scores = []
        for f in range(n_folds):
            if fold_size[f] >= max_size:
                scores.append(float("inf"))
                continue
            new_pos = fold_pos[f] + y[i]
            pos_err = np.sum(
                ((new_pos - target_pos) ** 2) / np.maximum(target_pos, 0.5)
            )
            size_err = ((fold_size[f] + 1 - target_size) ** 2) / max(target_size, 1)
            # Slight deterministic jitter prevents pathological repeated ties.
            scores.append(pos_err + 0.15 * size_err + 1e-7 * f)

        f = int(np.argmin(scores))
        assignment[i] = f
        fold_pos[f] += y[i]
        fold_size[f] += 1

    return assignment


expert = expert.copy()
expert["ft_fold"] = make_balanced_multilabel_folds(
    expert[TARGETS].to_numpy(np.float32),
    n_folds=FT_N_FOLDS,
    seed=SEED,
)

expert.to_csv(OUT / "expert_ft_fold_map.csv", index=False)

print("FT fold sizes:")
display(expert.groupby("ft_fold").size().rename("n").to_frame())

fold_pos = expert.groupby("ft_fold")[TARGETS].sum().T
fold_pos["TOTAL"] = expert[TARGETS].sum()
display(fold_pos)

## 7. Decode only the 58 expert studies, once per checkpoint pixel group

In [ ]:
train_series = pd.read_csv(
    ROOT / "train_series.csv",
    dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
)
plane_map = dict(zip(
    train_series["SeriesInstanceUID"].astype(str),
    train_series["Anatomical_Plane"],
))

expert_ids = expert["StudyInstanceUID"].astype(str).tolist()
htr = annotate(walk_selected("train_series", expert_ids))
log(f"expert header pass: {len(htr)} series across {htr['StudyInstanceUID'].nunique()} studies")

group_cache = {}

for gi, (pixel_key, gm) in enumerate(pixel_groups.items(), 1):
    cfg = json.loads(pixel_key)
    adopt_config_globals(cfg)
    log(
        f"decode group {gi}/{len(pixel_groups)}: "
        f"{IMG}px, {CACHE_SLICES} slices, group={GROUP}, crop={CROP_MM}mm"
    )

    slot_map = pick_slots(htr, plane_map)
    studies, cache, slot_mask = build_cache(
        slot_map,
        plane_map,
        lat_of(htr, f"expert g{gi} "),
        f"expert g{gi}",
    )

    group_cache[pixel_key] = {
        "studies": studies,
        "cache": cache,
        "slot_mask": slot_mask,
        "cfg": cfg,
    }

    log(
        f"g{gi}: cache {cache.shape}, slot coverage "
        f"{slot_mask.sum():.0f}/{slot_mask.size}"
    )

print("cached pixel groups:", len(group_cache))

## 8. Training utilities: masked balanced BCE, tiny LR, rollback gate

In [ ]:
def align_labels(studies):
    e = expert.set_index("StudyInstanceUID").loc[list(studies)]
    y = e[TARGETS].to_numpy(np.float32)
    mask = np.isfinite(y).astype(np.float32)
    y = np.nan_to_num(y, nan=0.0)
    folds = e["ft_fold"].to_numpy(np.int64)
    return y, mask, folds


def make_pos_weight(y, label_mask, idx):
    yy = y[idx]
    mm = label_mask[idx]
    pos = (yy * mm).sum(axis=0)
    known = mm.sum(axis=0)
    neg = known - pos

    # Conservative balancing: sqrt ratio, clipped to avoid huge gradients on rare targets.
    ratio = np.sqrt((neg + 0.5) / (pos + 0.5))
    ratio = np.clip(ratio, 1.0, 3.0)
    return torch.tensor(ratio, dtype=torch.float32)


def masked_bce_logits(logits, y, mask, pos_weight):
    per = F.binary_cross_entropy_with_logits(
        logits, y, reduction="none", pos_weight=pos_weight
    )
    return (per * mask).sum() / mask.sum().clamp_min(1.0)


def prob_bce(pred, y, mask):
    p = np.clip(np.asarray(pred, np.float64), 1e-6, 1 - 1e-6)
    y = np.asarray(y, np.float64)
    m = np.asarray(mask, np.float64)
    per = -(y * np.log(p) + (1 - y) * np.log(1 - p))
    return float((per * m).sum() / max(m.sum(), 1.0))


def macro_auc(pred, y, mask):
    vals = []
    per_target = {}
    for j, t in enumerate(TARGETS):
        valid = mask[:, j] > 0.5
        yy = y[valid, j]
        pp = pred[valid, j]
        if len(np.unique(yy)) < 2:
            per_target[t] = np.nan
            continue
        a = float(roc_auc_score(yy, pp))
        per_target[t] = a
        vals.append(a)
    return (float(np.mean(vals)) if vals else np.nan), per_target


def optimizer_for(model):
    head = [p for p in model.head.parameters() if p.requires_grad]
    backbone = [p for p in model.backbone.parameters() if p.requires_grad]

    return torch.optim.AdamW(
        [
            {"params": head, "lr": FT_HEAD_LR},
            {"params": backbone, "lr": FT_BACKBONE_LR},
        ],
        weight_decay=FT_WEIGHT_DECAY,
    )


def train_one_epoch(model, cache, slot_mask, y, label_mask, train_idx, dev, optimizer, scaler, pos_weight, epoch_seed):
    model.train()
    rng = np.random.default_rng(epoch_seed)
    order = np.asarray(train_idx, dtype=int).copy()
    rng.shuffle(order)

    optimizer.zero_grad(set_to_none=True)
    losses = []
    n_steps = math.ceil(len(order) / FT_BATCH)

    for step, b0 in enumerate(range(0, len(order), FT_BATCH), 1):
        sel = order[b0:b0 + FT_BATCH]
        starts = rng.integers(
            0,
            max(cache.shape[2] - GROUP + 1, 1),
            size=len(sel),
        )

        x_np = np.stack(
            [cache[i, :, int(st):int(st) + GROUP] for i, st in zip(sel, starts)],
            axis=0,
        )
        x = torch.from_numpy(np.ascontiguousarray(x_np)).to(dev)
        sm = torch.from_numpy(slot_mask[sel]).to(dev)
        yy = torch.from_numpy(y[sel]).to(dev)
        lm = torch.from_numpy(label_mask[sel]).to(dev)

        with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
            logits = model(x, sm, IMG)
            loss = masked_bce_logits(logits, yy, lm, pos_weight.to(dev))
            loss = loss / FT_GRAD_ACCUM

        scaler.scale(loss).backward()
        losses.append(float(loss.detach().cpu()) * FT_GRAD_ACCUM)

        if step % FT_GRAD_ACCUM == 0 or step == n_steps:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=1.0,
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

    return float(np.mean(losses)) if losses else np.nan


def cpu_state_dict(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

## 9. Conservative FT-A loop

Default behavior creates 5 new checkpoints: the best-holdout starting member from each original fold. A base member's fold number is used only to choose which **new expert fold** is held out during its FT-A stage.

The promotion rule starts from the original checkpoint as the best state. If fine-tuning never beats the base validation BCE, the saved FT-A checkpoint rolls back to the original weights.

In [ ]:
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log(f"device: {dev}")

metrics_rows = []
new_members = []

# For stage-level cross-validated diagnostics.
expert_uid_order = expert["StudyInstanceUID"].astype(str).tolist()
uid_to_global = {u: i for i, u in enumerate(expert_uid_order)}
oof_base_sum = np.zeros((len(expert), len(TARGETS)), np.float64)
oof_ft_sum = np.zeros_like(oof_base_sum)
oof_count = np.zeros(len(expert), np.int64)

for mi, m in enumerate(selected_members, 1):
    if time.time() - T0 > TIME_BUDGET:
        log("time budget reached before all selected members were processed")
        break

    pixel_key = m["pixel_group"]
    pack = group_cache[pixel_key]
    pixel_cfg = pack["cfg"]
    member_cfg = m.get("config") or {}
    adopt_config_globals(pixel_cfg)

    studies = pack["studies"]
    cache = pack["cache"]
    slot_mask = pack["slot_mask"]
    y, label_mask, ft_folds = align_labels(studies)

    # Historical fold ID is not a study mapping. We only use its 0..4 identity
    # to choose a distinct new expert-CV validation partition.
    val_fold = int(m.get("fold")) % FT_N_FOLDS
    train_idx = np.where(ft_folds != val_fold)[0]
    val_idx = np.where(ft_folds == val_fold)[0]

    if len(train_idx) == 0 or len(val_idx) == 0:
        raise RuntimeError(f"Empty train/val split for FT fold {val_fold}")

    log(
        f"[{mi}/{len(selected_members)}] {m['id']} base-fold={m.get('fold')} "
        f"-> FT-val-fold={val_fold}; train={len(train_idx)}, val={len(val_idx)}"
    )

    ck_path = WEIGHTS / m["file"]
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)

    model = build_model(
        FT_UNFREEZE_LAST,
        source=DINO,
        variant=member_cfg.get("variant", "small"),
        pool=member_cfg.get("pool", "cls_mean"),
        prior=bool(member_cfg.get("prior", False)),
    ).to(dev)

    model.load_state_dict(ck["model"], strict=True)

    if "fingerprint" in ck:
        check_fingerprint(model, dev, IMG, ck["fingerprint"], tag=f"{m['id']}: ")

    base_pred = predict_member_frontier(model, cache, slot_mask, val_idx, dev, IMG)
    base_loss = prob_bce(base_pred, y[val_idx], label_mask[val_idx])
    base_auc, _ = macro_auc(base_pred, y[val_idx], label_mask[val_idx])

    best_loss = base_loss
    best_auc = base_auc
    best_epoch = 0
    best_state = None
    best_pred = base_pred.copy()

    pos_weight = make_pos_weight(y, label_mask, train_idx)
    optimizer = optimizer_for(model)
    def lr_factor(ep):
        if FT_MAX_EPOCHS <= 1:
            return 1.0
        x = min(max(ep, 0), FT_MAX_EPOCHS)
        return max(0.10, 0.5 * (1.0 + math.cos(math.pi * x / FT_MAX_EPOCHS)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=[lr_factor, lr_factor]
    )
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

    stale = 0
    history = []

    for epoch in range(1, FT_MAX_EPOCHS + 1):
        train_loss = train_one_epoch(
            model,
            cache,
            slot_mask,
            y,
            label_mask,
            train_idx,
            dev,
            optimizer,
            scaler,
            pos_weight,
            epoch_seed=SEED + 1000 * mi + epoch,
        )

        val_pred = predict_member_frontier(
            model, cache, slot_mask, val_idx, dev, IMG
        )
        val_loss = prob_bce(val_pred, y[val_idx], label_mask[val_idx])
        val_auc, _ = macro_auc(val_pred, y[val_idx], label_mask[val_idx])

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_bce": val_loss,
            "val_macro_auc": val_auc,
            "head_lr": optimizer.param_groups[0]["lr"],
            "backbone_lr": optimizer.param_groups[1]["lr"],
        })

        log(
            f"  epoch {epoch}: train={train_loss:.5f}, "
            f"val_bce={val_loss:.5f}, val_auc={val_auc:.4f}"
        )

        if val_loss < best_loss - FT_MIN_DELTA:
            best_loss = val_loss
            best_auc = val_auc
            best_epoch = epoch
            best_state = cpu_state_dict(model)
            best_pred = val_pred.copy()
            stale = 0
        else:
            stale += 1

        scheduler.step()

        if stale >= FT_PATIENCE:
            log(f"  early stop after epoch {epoch}; no promotion for {stale} epoch(s)")
            break

    promoted = best_epoch > 0

    if promoted:
        model.load_state_dict(best_state, strict=True)
    else:
        model.load_state_dict(ck["model"], strict=True)
        best_loss = base_loss
        best_auc = base_auc
        best_pred = base_pred.copy()

    # Recompute a fingerprint for the exact state that will be handed to the next notebook.
    new_fp = fingerprint(model, dev, IMG).tolist()
    final_state = cpu_state_dict(model)

    new_id = f"{m['id']}_fta"
    new_file = f"fta_{m['id']}.pt"

    new_ck = dict(ck)
    new_ck["model"] = final_state
    new_ck["fingerprint"] = new_fp
    new_ck["ft_stage"] = "FT-A_CONSERVATIVE"
    new_ck["parent_id"] = m["id"]
    new_ck["parent_file"] = m["file"]
    new_ck["ft_validation_mode"] = "NEW_EXPERT_CV_NOT_HISTORICAL_OOF"
    new_ck["ft_val_fold"] = val_fold
    new_ck["ft_train_n"] = int(len(train_idx))
    new_ck["ft_val_n"] = int(len(val_idx))
    new_ck["ft_unfreeze_last"] = FT_UNFREEZE_LAST
    new_ck["ft_best_epoch"] = int(best_epoch)
    new_ck["ft_promoted"] = bool(promoted)
    new_ck["ft_base_val_bce"] = float(base_loss)
    new_ck["ft_best_val_bce"] = float(best_loss)
    new_ck["ft_base_val_macro_auc"] = float(base_auc) if np.isfinite(base_auc) else None
    new_ck["ft_best_val_macro_auc"] = float(best_auc) if np.isfinite(best_auc) else None
    new_ck["ft_history"] = history

    torch.save(new_ck, OUT / new_file)

    nm = dict(m)
    nm["id"] = new_id
    nm["file"] = new_file
    nm["ft_stage"] = "FT-A_CONSERVATIVE"
    nm["parent_id"] = m["id"]
    nm["parent_file"] = m["file"]
    nm["ft_val_fold"] = val_fold
    nm["ft_promoted"] = bool(promoted)
    nm["ft_best_epoch"] = int(best_epoch)
    nm["ft_base_val_bce"] = float(base_loss)
    nm["ft_best_val_bce"] = float(best_loss)
    nm["ft_best_val_macro_auc"] = float(best_auc) if np.isfinite(best_auc) else None
    new_members.append(nm)

    # Aggregate new expert-CV predictions. With champions mode count=1 per study;
    # with top2/all20 the same validation fold gets multiple diverse member votes.
    for local_pos, cache_idx in enumerate(val_idx):
        uid = studies[int(cache_idx)]
        g = uid_to_global[uid]
        oof_base_sum[g] += base_pred[local_pos]
        oof_ft_sum[g] += best_pred[local_pos]
        oof_count[g] += 1

    metrics_rows.append({
        "parent_id": m["id"],
        "new_id": new_id,
        "base_fold": int(m.get("fold")),
        "ft_val_fold": val_fold,
        "base_holdout": float(m.get("holdout") or np.nan),
        "train_n": len(train_idx),
        "val_n": len(val_idx),
        "base_val_bce": base_loss,
        "best_val_bce": best_loss,
        "delta_bce": best_loss - base_loss,
        "base_val_macro_auc": base_auc,
        "best_val_macro_auc": best_auc,
        "best_epoch": best_epoch,
        "promoted": promoted,
    })

    del model, ck, final_state, best_state
    gc.collect()
    if dev.type == "cuda":
        torch.cuda.empty_cache()

metrics = pd.DataFrame(metrics_rows)
metrics.to_csv(OUT / "ft_a_member_metrics.csv", index=False)
display(metrics)

## 10. Stage-level expert-CV diagnostic and handoff manifest

In [ ]:
valid = oof_count > 0
if not valid.all():
    missing_uids = expert.loc[~valid, "StudyInstanceUID"].tolist()
    print("WARNING: some expert studies received no validation prediction:", len(missing_uids))

base_oof = np.full_like(oof_base_sum, np.nan)
ft_oof = np.full_like(oof_ft_sum, np.nan)
base_oof[valid] = oof_base_sum[valid] / oof_count[valid, None]
ft_oof[valid] = oof_ft_sum[valid] / oof_count[valid, None]

oof_frame = expert[["StudyInstanceUID", "ft_fold", *TARGETS]].copy()
oof_frame["n_member_votes"] = oof_count

for j, t in enumerate(TARGETS):
    oof_frame[f"base__{t}"] = base_oof[:, j]
    oof_frame[f"fta__{t}"] = ft_oof[:, j]

oof_frame.to_csv(OUT / "expert_cv_predictions.csv", index=False)

if valid.any():
    y_all = expert.loc[valid, TARGETS].to_numpy(np.float32)
    mask_all = np.ones_like(y_all, np.float32)

    base_cv_bce = prob_bce(base_oof[valid], y_all, mask_all)
    ft_cv_bce = prob_bce(ft_oof[valid], y_all, mask_all)

    base_cv_auc, base_per = macro_auc(base_oof[valid], y_all, mask_all)
    ft_cv_auc, ft_per = macro_auc(ft_oof[valid], y_all, mask_all)

    target_diag = pd.DataFrame({
        "target": TARGETS,
        "base_auc": [base_per[t] for t in TARGETS],
        "fta_auc": [ft_per[t] for t in TARGETS],
    })
    target_diag["delta_auc"] = target_diag["fta_auc"] - target_diag["base_auc"]
    target_diag.to_csv(OUT / "expert_cv_target_auc.csv", index=False)

    print(f"Expert-CV base BCE: {base_cv_bce:.6f}")
    print(f"Expert-CV FT-A BCE: {ft_cv_bce:.6f}")
    print(f"Expert-CV base macro AUC: {base_cv_auc:.6f}")
    print(f"Expert-CV FT-A macro AUC: {ft_cv_auc:.6f}")
    display(target_diag)
else:
    base_cv_bce = ft_cv_bce = base_cv_auc = ft_cv_auc = np.nan

history = base_manifest.get("history")
if not isinstance(history, list):
    history = [] if history is None else [history]

stage2_history = history + [{
    "stage": "FT-A_CONSERVATIVE",
    "seed": SEED,
    "member_mode": FT_MEMBER_MODE,
    "n_output_members": len(new_members),
    "expert_n": len(expert),
    "validation_mode": "NEW_EXPERT_CV_NOT_HISTORICAL_OOF",
    "ft_unfreeze_last": FT_UNFREEZE_LAST,
    "max_epochs": FT_MAX_EPOCHS,
    "patience": FT_PATIENCE,
    "head_lr": FT_HEAD_LR,
    "backbone_lr": FT_BACKBONE_LR,
    "expert_cv_base_bce": None if not np.isfinite(base_cv_bce) else float(base_cv_bce),
    "expert_cv_fta_bce": None if not np.isfinite(ft_cv_bce) else float(ft_cv_bce),
    "expert_cv_base_macro_auc": None if not np.isfinite(base_cv_auc) else float(base_cv_auc),
    "expert_cv_fta_macro_auc": None if not np.isfinite(ft_cv_auc) else float(ft_cv_auc),
}]

new_manifest = {
    "format": base_manifest.get("format"),
    "targets": TARGETS,
    "history": stage2_history,
    "members": new_members,
}

(OUT / "manifest.json").write_text(json.dumps(new_manifest, indent=2))

status2 = {
    "stage2_ok": len(new_members) > 0,
    "stage": "FT-A_CONSERVATIVE",
    "member_mode": FT_MEMBER_MODE,
    "n_output_members": len(new_members),
    "n_promoted": int(metrics["promoted"].sum()) if len(metrics) else 0,
    "n_rolled_back_to_base": int((~metrics["promoted"]).sum()) if len(metrics) else 0,
    "validation_mode": "NEW_EXPERT_CV_NOT_HISTORICAL_OOF",
    "strict_historical_oof_available": False,
    "expert_cv_base_bce": None if not np.isfinite(base_cv_bce) else float(base_cv_bce),
    "expert_cv_fta_bce": None if not np.isfinite(ft_cv_bce) else float(ft_cv_bce),
    "expert_cv_base_macro_auc": None if not np.isfinite(base_cv_auc) else float(base_cv_auc),
    "expert_cv_fta_macro_auc": None if not np.isfinite(ft_cv_auc) else float(ft_cv_auc),
    "next_notebook": "03_hard_example_finetune.ipynb",
}
(OUT / "stage2_status.json").write_text(json.dumps(status2, indent=2))

print(json.dumps(status2, indent=2))

## 11. Output inventory

Do **not** submit anything from this notebook to the competition yet. This stage produces new model weights, not the final submission.

For Notebook 03, add this notebook's Kaggle output as an input. The important files are:

- `rsna_stage2_fta/manifest.json`
- `rsna_stage2_fta/fta_*.pt`
- `rsna_stage2_fta/ft_a_member_metrics.csv`
- `rsna_stage2_fta/expert_cv_predictions.csv`
- `rsna_stage2_fta/expert_cv_target_auc.csv`
- `rsna_stage2_fta/expert_ft_fold_map.csv`
- `rsna_stage2_fta/stage2_status.json`

The original 20 checkpoints remain untouched and will return in the final ensemble.

In [ ]:
print("Stage 2 output:")
total_bytes = 0
for p in sorted(OUT.iterdir()):
    if p.is_file():
        total_bytes += p.stat().st_size
        print(f" - {p.name:40s} {p.stat().st_size / (1024**2):9.2f} MB")

print(f"Total output size: {total_bytes / (1024**3):.3f} GB")
print("Next: add this notebook output as a Kaggle input to Notebook 03.")